# OccluNet: Occlusion-Robust Road Extraction + Network Resilience

## Architecture Summary
```
INPUT (4ch: RGB + NIR)
  → ResNet34 Encoder + CBAM at each scale
  → Dilated Bottleneck (rates 1,2,4,8)
  → UNet Decoder + RCPM at 64x64 scale (BiGRU path continuity)
  → Head A: Segmentation mask
  → Head B: Skeleton centerline
  → Graph Pipeline: sknw → MST healing → Betweenness Centrality → Resilience Index
  → Gradio 5-panel Dashboard
```

## Import Essentials

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'segmentation-models-pytorch',
    'timm',
    'rasterio',
    'shapely',
    'fiona',
    'geopandas',
    'sknw',
    'networkx',
    'scikit-image',
    'scipy',
    'gradio',
    'matplotlib',
    'tqdm',
    'opencv-python-headless',
])
print('✅ Dependencies installed')

In [ ]:
import os, sys, torch
SPACENET_ROOT  = '/kaggle/input/datasets/tejaskhanna007/spacenet-5-mumbai-roads/nfs/data/cosmiq/spacenet/competitions/SN5_roads/tiles_upload/train/AOI_8_Mumbai'   # folder containing AOI_8_Mumbai
SENTINEL_ROOT  = '/kaggle/input/datasets/tejaskhanna007/sentinel-2-validation-data'  # folder with Bengaluru chips
OUTPUT_DIR     = '/kaggle/working/occlunet'

BATCH_SIZE     = 4
IMG_SIZE       = 512
PHASE1_EPOCHS  = 40
PHASE2_EPOCHS  = 30
PHASE3_EPOCHS  = 20
NUM_WORKERS    = 2

os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO GPU ')

def find_spacenet_paths(root):
    import glob
    tif_pattern = os.path.join(root, '**', '*.tif')
    tifs = glob.glob(tif_pattern, recursive=True)
    geojson_pattern = os.path.join(root, '**', '*.geojson')
    geojsons = glob.glob(geojson_pattern, recursive=True)
    print(f'Found {len(tifs)} TIF files and {len(geojsons)} GeoJSON files')
    return tifs, geojsons

if os.path.exists(SPACENET_ROOT):
    tifs, geojsons = find_spacenet_paths(SPACENET_ROOT)
else:
    print(f'SpaceNet root not found: {SPACENET_ROOT}')
    print('   → Update SPACENET_ROOT above to match your dataset')
    tifs, geojsons = [], []

## Data Verification

In [ ]:
import os
import re
import random
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import json
from rasterio.features import rasterize
from scipy.ndimage import binary_dilation
from skimage.morphology import skeletonize
import warnings
warnings.filterwarnings('ignore')

def load_ps_ms_chip(tif_path):
    """Load SpaceNet PS-MS chip. Returns (H,W,4) array: R,G,B,NIR"""
    try:
        with rasterio.open(tif_path) as src:
            data = src.read() 
            profile = src.profile
            transform = src.transform
            crs = src.crs
    except Exception as e:
        print(f"Rasterio failed to read file: {tif_path}. Error: {e}")
        return None, None, None, None

    num_bands = data.shape[0]
    
    if num_bands >= 8:
        r   = data[4].astype(np.float32)  
        g   = data[2].astype(np.float32) 
        b   = data[1].astype(np.float32) 
        nir = data[6].astype(np.float32) 
    elif num_bands >= 4:
        r, g, b, nir = [data[i].astype(np.float32) for i in range(4)]
    elif num_bands == 3:
        r, g, b = [data[i].astype(np.float32) for i in range(3)]
        nir = r.copy() 
    else:
        r = data[0].astype(np.float32)
        g = r.copy()
        b = r.copy()
        nir = r.copy()
    
    chip = np.stack([r, g, b, nir], axis=-1)  # (H,W,4)
    return chip, profile, transform, crs

def percentile_stretch(arr):
    out = np.zeros_like(arr, dtype=np.float32)
    for c in range(arr.shape[-1]):
        ch = arr[..., c]
        p2, p98 = np.percentile(ch, 2), np.percentile(ch, 98)
        out[..., c] = np.clip((ch - p2) / (p98 - p2 + 1e-6), 0, 1)
    return out

def load_road_mask(geojson_path, transform, height, width, dilation_px=3):
    with open(geojson_path) as f:
        gj = json.load(f)
    features = gj.get('features', [])
    if not features:
        return np.zeros((height, width), dtype=np.uint8)
    shapes = [(feat['geometry'], 1) for feat in features if feat.get('geometry') is not None]
    if not shapes:
        return np.zeros((height, width), dtype=np.uint8)
    mask = rasterize(shapes, out_shape=(height, width), transform=transform, fill=0, dtype=np.uint8, all_touched=True)
    if dilation_px > 0:
        struct = np.ones((dilation_px*2+1, dilation_px*2+1), dtype=bool)
        mask = binary_dilation(mask, structure=struct).astype(np.uint8)
    return mask

def match_geojson(tif_path, geojsons):
    """Corrected regex matcher looking for 'chip<N>' or 'img<N>' variations"""
    tif_stem = os.path.splitext(os.path.basename(tif_path))[0]
    
    num_match = re.search(r'(chip|img)(\d+)', tif_stem, re.IGNORECASE)
    if not num_match:
        return None
        
    chip_num = num_match.group(2)
    
    for gj in geojsons:
        gj_stem = os.path.splitext(os.path.basename(gj))[0]
        gj_match = re.search(r'(chip|img)(\d+)', gj_stem, re.IGNORECASE)
        if gj_match and gj_match.group(2) == chip_num:
            return gj
    return None

if tifs and geojsons:
   
    valid_tifs = [
        t for t in tifs 
        if "ps-ms" in t.lower() or "ms_" in t.lower()
    ]
    
    if not valid_tifs:
        print(" No matching multi-spectral chips found, falling back to all available TIFs.")
        valid_tifs = tifs

    sample_tifs = random.sample(valid_tifs, min(5, len(valid_tifs)))
    verified = 0
    
    fig, axes = plt.subplots(len(sample_tifs), 3, figsize=(15, 5*len(sample_tifs)))
    if len(sample_tifs) == 1:
        axes = [axes]
    
    for i, tif in enumerate(sample_tifs):
        chip, profile, transform, crs = load_ps_ms_chip(tif)
        if chip is None:
            continue
            
        chip_norm = percentile_stretch(chip)
        H, W = chip.shape[:2]
        
        gj_path = match_geojson(tif, geojsons)
        if gj_path is None:
            print(f'[{i}]  No matching GeoJSON found for: {os.path.basename(tif)}')
            continue
        
        seg_mask = load_road_mask(gj_path, transform, H, W, dilation_px=3)
        skel_mask = load_road_mask(gj_path, transform, H, W, dilation_px=1)
        
        if seg_mask.sum() > 0:
            skel_mask = skeletonize(skel_mask).astype(np.uint8)
            verified += 1
        else:
            skel_mask = np.zeros_like(seg_mask)
        
        road_pixels = seg_mask.sum()
        skel_pixels = skel_mask.sum()
        
        print(f'[{i}] Successfully Paired & Loaded: {os.path.basename(tif)}')
        print(f'     Shape: {chip.shape} | Road Vector Pixels: {road_pixels}')
        
        axes[i][0].imshow(chip_norm[..., :3])
        axes[i][0].set_title(f'RGB chip {i}')
        axes[i][0].axis('off')
        axes[i][1].imshow(seg_mask, cmap='Reds')
        axes[i][1].set_title(f'Seg mask | sum={road_pixels}')
        axes[i][1].axis('off')
        axes[i][2].imshow(skel_mask, cmap='Blues')
        axes[i][2].set_title(f'Skeleton GT | sum={skel_pixels}')
        axes[i][2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/data_verification.png', dpi=100)
    plt.show()
    print(f'\n Validated Pairs: {verified}/{len(sample_tifs)} contain active labels.')
else:
    print(' No TIFs or GeoJSONs found. Check SPACENET_ROOT path.')

## Augmentation and Loading

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import random
import cv2
import os

class MumbaiRoadsDataset(Dataset):
    """
    Returns (x_4ch, y_seg, y_skel) triplets.
    x_4ch: (4, 512, 512) float32 in [0,1] — R, G, B, NIR
    y_seg:  (1, 512, 512) float32 binary — dilated road mask (3px)
    y_skel: (1, 512, 512) float32 binary — skeleton centerline (1px)
    """
    def __init__(self, tif_paths, geojson_list, img_size=512, augment=True):
        self.img_size = img_size
        self.augment = augment
        
        valid_tifs = [
            t for t in tif_paths 
            if "ps-ms" in t.lower() or "ms_" in t.lower()
        ]
        if not valid_tifs:
            valid_tifs = tif_paths
            
        self.pairs = []
        for tif in valid_tifs:
            gj = match_geojson(tif, geojson_list)
            if gj is not None:
                self.pairs.append((tif, gj))
        
        print(f'Dataset: {len(self.pairs)} matched chip-GeoJSON pairs successfully initialized.')
        if len(self.pairs) == 0:
            raise ValueError('No matched pairs found! Check your folder structure or regex patterns.')
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        tif_path, gj_path = self.pairs[idx]
        
        chip, profile, transform, crs = load_ps_ms_chip(tif_path)
        H, W = chip.shape[:2]
        
        seg_mask  = load_road_mask(gj_path, transform, H, W, dilation_px=3)
        skel_mask = load_road_mask(gj_path, transform, H, W, dilation_px=1)
        
        if seg_mask.sum() > 0:
            from skimage.morphology import skeletonize
            skel_mask = skeletonize(skel_mask).astype(np.uint8)
        else:
            skel_mask = np.zeros_like(seg_mask)
        
        chip     = cv2.resize(chip,     (self.img_size, self.img_size),
                              interpolation=cv2.INTER_LINEAR)
        seg_mask = cv2.resize(seg_mask, (self.img_size, self.img_size),
                              interpolation=cv2.INTER_NEAREST)
        skel_mask = cv2.resize(skel_mask, (self.img_size, self.img_size),
                               interpolation=cv2.INTER_NEAREST)
        
        chip = percentile_stretch(chip) 
        
        x = torch.from_numpy(chip.transpose(2, 0, 1)).float()
        y_seg  = torch.from_numpy(seg_mask.astype(np.float32)).unsqueeze(0)
        y_skel = torch.from_numpy(skel_mask.astype(np.float32)).unsqueeze(0)
        
        if self.augment:
            x, y_seg, y_skel = self._augment(x, y_seg, y_skel)
        
        return x, y_seg, y_skel
    
    def _augment(self, x, y_seg, y_skel):
        if random.random() > 0.5:
            x, y_seg, y_skel = TF.hflip(x), TF.hflip(y_seg), TF.hflip(y_skel)
        if random.random() > 0.5:
            x, y_seg, y_skel = TF.vflip(x), TF.vflip(y_seg), TF.vflip(y_skel)
        if random.random() > 0.7:
            k = random.choice([1, 2, 3])
            x = torch.rot90(x, k, [1, 2])
            y_seg = torch.rot90(y_seg, k, [1, 2])
            y_skel = torch.rot90(y_skel, k, [1, 2])
            
        
        if random.random() > 0.6:
            n_patches = random.randint(1, 4)
            H, W = x.shape[1], x.shape[2]
            for _ in range(n_patches):
                ph, pw = random.randint(20, 80), random.randint(20, 80)
                py, px = random.randint(0, H - ph), random.randint(0, W - pw)
                x[:, py:py+ph, px:px+pw] = 0.0  
                
        if random.random() > 0.75:
            
            gray = 0.299 * x[0] + 0.587 * x[1] + 0.114 * x[2]
            x[0] = gray
            x[1] = gray
            x[2] = gray
        else:
            
            if random.random() > 0.5:
                rgb = x[:3]
                
                factor_b = random.uniform(0.6, 1.4)
                
                mean_c = rgb.mean(dim=[1, 2], keepdim=True)
                factor_c = random.uniform(0.7, 1.3)
                
                rgb = (rgb * factor_b - mean_c) * factor_c + mean_c
                x[:3] = rgb.clamp(0, 1)
        
        return x, y_seg, y_skel
if 'tifs' in globals() and 'geojsons' in globals() and tifs and geojsons:
    from torch.utils.data import random_split
    
    full_ds = MumbaiRoadsDataset(tifs, geojsons, img_size=IMG_SIZE, augment=True)
    
    n_val = max(20, int(0.1 * len(full_ds)))
    n_train = len(full_ds) - n_val
    train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=NUM_WORKERS,
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS,
                              pin_memory=True)
    
    x_b, y_seg_b, y_skel_b = next(iter(train_loader))
    print(f'successfully initialized train_loader!')
    print(f'Batch shapes: x={x_b.shape}, y_seg={y_seg_b.shape}, y_skel={y_skel_b.shape}')
else:
    print('Cannot create loaders: tifs or geojsons variables are empty or missing!')

## Model: OccluNet (ResNet34 + CBAM + Dilated Bottleneck + RCPM + Dual Heads)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg = self.mlp(self.avg_pool(x))
        mx  = self.mlp(self.max_pool(x))
        return self.sigmoid(avg + mx)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        spatial = torch.cat([avg, mx], dim=1)
        return self.sigmoid(self.conv(spatial))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention()
    
    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

class DilatedBottleneck(nn.Module):
    def __init__(self, channels=512):
        super().__init__()
        mid = channels // 4
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, mid, 3, padding=r, dilation=r, bias=False),
                nn.BatchNorm2d(mid), nn.ReLU(inplace=True)
            ) for r in [1, 2, 4, 8]
        ])
        self.project = nn.Sequential(
            nn.Conv2d(mid * 4, channels, 1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        outs = [branch(x) for branch in self.branches]
        return self.project(torch.cat(outs, dim=1))

class RCPM(nn.Module):
    def __init__(self, channels=128):
        super().__init__()
        self.strip_h = nn.Sequential(
            nn.Conv2d(channels, channels, (1, 15), padding=(0, 7), groups=channels, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True)
        )
        self.strip_v = nn.Sequential(
            nn.Conv2d(channels, channels, (15, 1), padding=(7, 0), groups=channels, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True)
        )
        self.gru_h = nn.GRU(channels, channels // 2, batch_first=True, bidirectional=True)
        self.gru_w = nn.GRU(channels, channels // 2, batch_first=True, bidirectional=True)
        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        B, C, H, W = x.shape
        sh = self.strip_h(x)
        sv = self.strip_v(x)
        
        x_h = x.permute(0, 2, 3, 1).reshape(B * H, W, C)
        out_h, _ = self.gru_h(x_h)
        out_h = out_h.reshape(B, H, W, C).permute(0, 3, 1, 2)
        
        x_w = x.permute(0, 3, 2, 1).reshape(B * W, H, C)
        out_w, _ = self.gru_w(x_w)
        out_w = out_w.reshape(B, W, H, C).permute(0, 3, 2, 1)
        
        fused = torch.cat([sh + sv, out_h, out_w], dim=1)
        return self.fusion(fused)

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, use_rcpm=False):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        merged_ch = in_ch // 2 + skip_ch
        self.conv = nn.Sequential(
            nn.Conv2d(merged_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
        self.rcpm = RCPM(out_ch) if use_rcpm else None
    
    def forward(self, x, skip):
        x = self.up(x)
        diffH = skip.shape[2] - x.shape[2]
        diffW = skip.shape[3] - x.shape[3]
        x = F.pad(x, [diffW//2, diffW-diffW//2, diffH//2, diffH-diffH//2])
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        if self.rcpm is not None:
            x = x + self.rcpm(x)
        return x

class OccluNetLite(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        resnet = models.resnet34(weights='IMAGENET1K_V1')
        
        old_conv = resnet.conv1
        new_conv = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3]  = old_conv.weight[:, 0] # Initialize NIR with Red weights
            
        self.enc0 = nn.Sequential(new_conv, resnet.bn1, resnet.relu) # Out: 64ch, 256x256
        self.pool  = resnet.maxpool                                 # Out: 64ch, 128x128
        self.enc1  = resnet.layer1                                  # Out: 64ch, 128x128
        self.enc2  = resnet.layer2                                  # Out: 128ch, 64x64
        self.enc3  = resnet.layer3                                  # Out: 256ch, 32x32
        self.enc4  = resnet.layer4                                  # Out: 512ch, 16x16
        
        self.cbam1 = CBAM(64)
        self.cbam2 = CBAM(128)
        self.cbam3 = CBAM(256)
        self.cbam4 = CBAM(512)
        
        self.bottleneck = DilatedBottleneck(512)
        
        self.dec4 = DecoderBlock(512, 256, 256, use_rcpm=False)    
        self.dec3 = DecoderBlock(256, 128, 128, use_rcpm=True)     
        self.dec2 = DecoderBlock(128, 64, 64, use_rcpm=False)       
        self.dec1 = DecoderBlock(64, 64, 32, use_rcpm=False)       
        
        self.dec0 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 2, stride=2),               
            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )
        
        self.seg_head  = nn.Conv2d(16, 1, 1)
        self.skel_head = nn.Conv2d(16, 1, 1)
    
    def forward(self, x):
        e0 = self.enc0(x)                
        e0p = self.pool(e0)              
        e1 = self.cbam1(self.enc1(e0p))  
        e2 = self.cbam2(self.enc2(e1))   
        e3 = self.cbam3(self.enc3(e2))   
        e4 = self.cbam4(self.enc4(e3))   
        
        b = self.bottleneck(e4)
        
        d4 = self.dec4(b, e3)           
        d3 = self.dec3(d4, e2)           
        d2 = self.dec2(d3, e1)           
        d1 = self.dec1(d2, e0)           
        d0 = self.dec0(d1)               
        
        seg_logits  = self.seg_head(d0)
        skel_logits = self.skel_head(d0)
        
        return seg_logits, skel_logits
    
    def get_param_groups(self):
        enc_early = list(self.enc0.parameters()) + list(self.enc1.parameters())
        enc_late  = list(self.enc2.parameters()) + list(self.enc3.parameters()) + list(self.enc4.parameters())
        rest_params = set()
        for p in self.parameters():
            if not any(p is q for q in enc_early + enc_late):
                rest_params.add(p)
        return enc_early, enc_late, list(rest_params)


model = OccluNetLite(in_channels=4).to(DEVICE)
with torch.no_grad():
    dummy = torch.zeros(2, 4, IMG_SIZE, IMG_SIZE).to(DEVICE)
    seg_out, skel_out = model(dummy)

print(f'seg_out shape:  {seg_out.shape}   ' if seg_out.shape == (2,1,512,512) else f' {seg_out.shape}')
print(f'skel_out shape: {skel_out.shape}  ' if skel_out.shape == (2,1,512,512) else f' {skel_out.shape}')

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {total_params/1e6:.1f}M')

## Loss Functions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2 * (probs * targets).sum() + self.smooth
        den = probs.sum() + targets.sum() + self.smooth
        return 1 - num / den


class WeightedBCELoss(nn.Module):
    """Handles class imbalance: road pixels (~5%) vs background (~95%)."""
    def __init__(self, pos_weight=15.0):
        super().__init__()
        self.pos_weight = pos_weight
    
    def forward(self, logits, targets):
        pw = torch.tensor(self.pos_weight, device=logits.device)
        return F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=pw
        )


class SoftSkeletonize(nn.Module):
    """Differentiable skeletonization via iterative morphological erosion."""
    def __init__(self, num_iter=40):
        super().__init__()
        self.num_iter = num_iter
    
    def soft_erode(self, img):
        if img.dim() == 3:
            img = img.unsqueeze(0)
        B, C, H, W = img.shape
        p1 = -F.max_pool2d(-img, (3,1), (1,1), (1,0))
        p2 = -F.max_pool2d(-img, (1,3), (1,1), (0,1))
        return torch.min(p1, p2)
    
    def soft_dilate(self, img):
        if img.dim() == 3:
            img = img.unsqueeze(0)
        return F.max_pool2d(img, (3,3), (1,1), (1,1))
    
    def soft_open(self, img):
        return self.soft_dilate(self.soft_erode(img))
    
    def forward(self, img):
        skel = torch.zeros_like(img)
        delta = 1.0
        for _ in range(self.num_iter):
            eroded = self.soft_erode(img)
            delta = img - self.soft_open(img)
            skel = skel + delta * img
            img = eroded
        return skel


class CLDiceLoss(nn.Module):
    """
    Centerline Dice Loss (Shit et al. 2021).
    Penalizes topological breaks in road network.
    Directly addresses requirement for 'routable topology'.
    """
    def __init__(self, iter_=40, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.soft_skel = SoftSkeletonize(num_iter=iter_)
    
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        skel_pred   = self.soft_skel(probs)
        skel_target = self.soft_skel(targets)
        
        tprec = ((skel_pred * targets).sum() + self.smooth) / \
                (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * probs).sum() + self.smooth) / \
                (skel_target.sum() + self.smooth)
        
        cl_dice = 1 - 2 * tprec * tsens / (tprec + tsens + 1e-6)
        return cl_dice


class ConformityLoss(nn.Module):
    """
    Enforces skeleton ⊆ segmentation mask.
    Prevents phantom centerlines outside road body.
    """
    def forward(self, seg_logits, skel_logits):
        seg_probs  = torch.sigmoid(seg_logits)
        skel_probs = torch.sigmoid(skel_logits)
        return F.relu(skel_probs - seg_probs).mean()


class FullLoss(nn.Module):
    """
    Combined loss for both heads.
    
    Phase 1 (seg only):   L = Dice(0.4) + WBCE(0.6)
    Phase 2 (both heads): L = Dice(0.35) + WBCE(0.35) + CLDice(0.3)
                            + 0.3 * L_skel + 0.15 * L_conf
    """
    def __init__(self, phase=1):
        super().__init__()
        self.phase    = phase
        self.dice     = DiceLoss()
        self.wbce_seg = WeightedBCELoss(pos_weight=15.0)
        self.wbce_sk  = WeightedBCELoss(pos_weight=30.0)
        self.cldice   = CLDiceLoss()
        self.conform  = ConformityLoss()
    
    def forward(self, seg_logits, skel_logits, y_seg, y_skel):
        if self.phase == 1:
            l = 0.4 * self.dice(seg_logits, y_seg) + \
                0.6 * self.wbce_seg(seg_logits, y_seg)
            return l, {'seg': l.item(), 'total': l.item()}
        else:
            l_seg = (0.35 * self.dice(seg_logits, y_seg) +
                     0.35 * self.wbce_seg(seg_logits, y_seg) +
                     0.30 * self.cldice(seg_logits, y_seg))
            
            l_skel = (0.5  * self.dice(skel_logits, y_skel) +
                      0.5  * self.wbce_sk(skel_logits, y_skel))
            
            l_conf = self.conform(seg_logits, skel_logits)
            
            total = l_seg + 0.3 * l_skel + 0.15 * l_conf
            return total, {
                'seg':   l_seg.item(),
                'skel':  l_skel.item(),
                'conf':  l_conf.item(),
                'total': total.item()
            }


dummy_logits = torch.randn(2, 1, 512, 512)
dummy_targets = (torch.rand(2, 1, 512, 512) > 0.9).float()  # ~10% road

criterion = FullLoss(phase=1)
loss, info = criterion(dummy_logits, dummy_logits, dummy_targets, dummy_targets)
print(f'Phase 1 test loss: {loss.item():.4f} ' if 0 < loss.item() < 5 else f' {loss.item()}')

criterion2 = FullLoss(phase=2)
loss2, info2 = criterion2(dummy_logits, dummy_logits, dummy_targets, dummy_targets)
print(f'Phase 2 test loss: {loss2.item():.4f} ' if 0 < loss2.item() < 5 else f'3{loss2.item()}')
print(f'Loss components: {info2}')

## Metrics & Training Utilities

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt


def compute_metrics(logits, targets, threshold=0.5):
    """Returns IoU, Dice, Recall, Precision."""
    probs = torch.sigmoid(logits).detach().cpu()
    preds = (probs > threshold).float()
    targs = targets.detach().cpu().float()
    
    TP = (preds * targs).sum().item()
    FP = (preds * (1 - targs)).sum().item()
    FN = ((1 - preds) * targs).sum().item()
    
    iou       = TP / (TP + FP + FN + 1e-6)
    dice      = 2 * TP / (2 * TP + FP + FN + 1e-6)
    recall    = TP / (TP + FN + 1e-6)
    precision = TP / (TP + FP + 1e-6)
    
    return {'iou': iou, 'dice': dice, 'recall': recall, 'precision': precision}


def visualize_prediction(model, val_loader, device, epoch, output_dir, phase=1):
    """Visualize one prediction to monitor training progress."""
    model.eval()
    with torch.no_grad():
        x, y_seg, y_skel = next(iter(val_loader))
        x = x.to(device)
        seg_logits, skel_logits = model(x)
        seg_prob  = torch.sigmoid(seg_logits[0, 0]).cpu().numpy()
        skel_prob = torch.sigmoid(skel_logits[0, 0]).cpu().numpy()
        
        fig, axes = plt.subplots(1, 5 if phase >= 2 else 4, figsize=(20, 4))
        
        rgb = x[0, :3].cpu().permute(1, 2, 0).numpy().clip(0, 1)
        axes[0].imshow(rgb)
        axes[0].set_title('RGB Input')
        axes[0].axis('off')
        
        axes[1].imshow(y_seg[0, 0].numpy(), cmap='Reds')
        axes[1].set_title('GT Seg')
        axes[1].axis('off')
        
        axes[2].imshow(seg_prob, cmap='Reds', vmin=0, vmax=1)
        axes[2].set_title(f'Pred Seg (ep {epoch})')
        axes[2].axis('off')
        
        axes[3].imshow((seg_prob > 0.5).astype(float), cmap='gray')
        axes[3].set_title('Pred > 0.5')
        axes[3].axis('off')
        
        if phase >= 2:
            axes[4].imshow(skel_prob, cmap='Blues', vmin=0, vmax=1)
            axes[4].set_title(f'Pred Skeleton')
            axes[4].axis('off')
        
        plt.suptitle(f'Epoch {epoch}', fontsize=14)
        plt.tight_layout()
        save_path = f'{output_dir}/pred_epoch_{epoch:03d}.png'
        plt.savefig(save_path, dpi=100)
        plt.show()
        print(f'Saved visualization: {save_path}')
    
    model.train()


def early_stop_diagnostic(val_iou, epoch):
    """Mandatory diagnostic at every 5th epoch."""
    if epoch % 5 != 0:
        return True  
    
    print(f'\n── Diagnostic @ epoch {epoch} ──')
    print(f'   val_IoU = {val_iou:.4f}')
    
    if epoch == 10 and val_iou < 0.15:
        print('    STOP: val_IoU < 0.15 at epoch 10 → data pipeline broken')
        return False
    if epoch == 15 and val_iou < 0.20:
        print('     WARNING: val_IoU < 0.20 at epoch 15 → check normalization')
    if val_iou > 0.40:
        print(f'    val_IoU > 0.40 achieved!')
    
    return True


print(' Metrics and utilities defined')

## Phase 1 Training (40 epochs, seg head only)

In [ ]:
import torch.optim as optim
from tqdm import tqdm

enc_early_params, enc_late_params, rest_params = model.get_param_groups()

optimizer_p1 = optim.AdamW([
    {'params': enc_early_params, 'lr': 0.0},     
    {'params': enc_late_params,  'lr': 5e-5},     
    {'params': rest_params,      'lr': 1e-4},      
], weight_decay=1e-4)

scheduler_p1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p1, mode='max', patience=5, factor=0.5
)

criterion_p1 = FullLoss(phase=1).to(DEVICE)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

best_val_iou = 0.0
best_ckpt_path = f'{OUTPUT_DIR}/best_phase1_robust.pth'
history = {'train_loss': [], 'val_iou': [], 'val_dice': [], 'val_recall': []}

print('Starting Phase 1 training (seg head, 40 epochs)...')
print('Expected: val_IoU > 0.40 by epoch 30')
print()

for epoch in range(1, PHASE1_EPOCHS + 1):
    if epoch == 11:
        optimizer_p1.param_groups[0]['lr'] = 1e-5
        print('→ Unfreezing encoder layers 1-2 (LR = 1e-5)')
    
    
    model.train()
    train_loss = 0.0
    
    for x, y_seg, y_skel in tqdm(train_loader, desc=f'Ep{epoch:03d} train',
                                  leave=False):
        x      = x.to(DEVICE)
        y_seg  = y_seg.to(DEVICE)
        y_skel = y_skel.to(DEVICE)
        
        optimizer_p1.zero_grad()
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                seg_logits, skel_logits = model(x)
                loss, _ = criterion_p1(seg_logits, skel_logits, y_seg, y_skel)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_p1)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer_p1)
            scaler.update()
        else:
            seg_logits, skel_logits = model(x)
            loss, _ = criterion_p1(seg_logits, skel_logits, y_seg, y_skel)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_p1.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    model.eval()
    val_metrics = {'iou': 0, 'dice': 0, 'recall': 0, 'precision': 0}
    n_val_batches = 0
    
    with torch.no_grad():
        for x, y_seg, y_skel in val_loader:
            x = x.to(DEVICE)
            y_seg = y_seg.to(DEVICE)
            seg_logits, _ = model(x)
            m = compute_metrics(seg_logits, y_seg)
            for k in val_metrics:
                val_metrics[k] += m[k]
            n_val_batches += 1
    
    for k in val_metrics:
        val_metrics[k] /= n_val_batches
    
    val_iou = val_metrics['iou']
    scheduler_p1.step(val_iou)
    
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'val_iou': val_iou,
            'phase': 1
        }, best_ckpt_path)
    
    history['train_loss'].append(train_loss)
    history['val_iou'].append(val_iou)
    history['val_dice'].append(val_metrics['dice'])
    history['val_recall'].append(val_metrics['recall'])
    
    print(f'Ep{epoch:03d} | loss={train_loss:.4f} | '
          f'val_IoU={val_iou:.4f} | dice={val_metrics["dice"]:.4f} | '
          f'recall={val_metrics["recall"]:.4f} | '
          f'best={best_val_iou:.4f}')
    
    if not early_stop_diagnostic(val_iou, epoch):
        print('Training stopped by diagnostic. Fix data pipeline.')
        break
    
    if epoch in [5, 10, 20, 30, PHASE1_EPOCHS]:
        visualize_prediction(model, val_loader, DEVICE, epoch, OUTPUT_DIR, phase=1)

print(f'\n Phase 1 complete. Best val_IoU: {best_val_iou:.4f}')
print(f'   Best checkpoint: {best_ckpt_path}')

## Phase 2 Training (30 epochs,  both heads)

In [ ]:
import torch.optim as optim
from tqdm import tqdm
best_ckpt_path = f'{OUTPUT_DIR}/best_phase1_robust.pth'
enc_early_params, enc_late_params, rest_params = model.get_param_groups()
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded Phase 1 checkpoint from epoch {ckpt["epoch"]} (IoU={ckpt["val_iou"]:.4f})')

optimizer_p2 = optim.AdamW([
    {'params': enc_early_params, 'lr': 1e-6},
    {'params': enc_late_params,  'lr': 5e-6},
    {'params': rest_params,      'lr': 1e-5},
], weight_decay=1e-4)

scheduler_p2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=PHASE2_EPOCHS, eta_min=1e-7
)

criterion_p2 = FullLoss(phase=2).to(DEVICE)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

best_val_iou_p2 = 0.0
best_ckpt_p2 = f'{OUTPUT_DIR}/best_phase2_robust.pth'
history_p2 = {'train_loss': [], 'val_iou': []}

print('Starting Phase 2 training (both heads + CLDice, 30 epochs)...')
print('Expected: val_IoU > 0.48, skeleton head producing visible centerlines')
print()

for epoch in range(PHASE1_EPOCHS + 1, PHASE1_EPOCHS + PHASE2_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    
    for x, y_seg, y_skel in tqdm(train_loader, desc=f'Ep{epoch:03d} P2',
                                  leave=False):
        x      = x.to(DEVICE)
        y_seg  = y_seg.to(DEVICE)
        y_skel = y_skel.to(DEVICE)
        
        optimizer_p2.zero_grad()
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                seg_logits, skel_logits = model(x)
                loss, info = criterion_p2(seg_logits, skel_logits, y_seg, y_skel)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer_p2)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer_p2)
            scaler.update()
        else:
            seg_logits, skel_logits = model(x)
            loss, info = criterion_p2(seg_logits, skel_logits, y_seg, y_skel)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_p2.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    scheduler_p2.step()
    
    model.eval()
    val_ious, n = 0.0, 0
    with torch.no_grad():
        for x, y_seg, y_skel in val_loader:
            x = x.to(DEVICE)
            y_seg = y_seg.to(DEVICE)
            seg_logits, skel_logits = model(x)
            m = compute_metrics(seg_logits, y_seg)
            val_ious += m['iou']
            n += 1
    
    val_iou = val_ious / n
    
    if val_iou > best_val_iou_p2:
        best_val_iou_p2 = val_iou
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'val_iou': val_iou,
            'phase': 2
        }, best_ckpt_p2)
    
    print(f'Ep{epoch:03d} | loss={train_loss:.4f} | '
          f'val_IoU={val_iou:.4f} | best_P2={best_val_iou_p2:.4f}')
    
    if epoch in [PHASE1_EPOCHS + 5, PHASE1_EPOCHS + 15, PHASE1_EPOCHS + PHASE2_EPOCHS]:
        visualize_prediction(model, val_loader, DEVICE, epoch, OUTPUT_DIR, phase=2)

print(f'\n Phase 2 complete. Best val_IoU: {best_val_iou_p2:.4f}')
print(f'   Best checkpoint: {best_ckpt_p2}')

## Deep Globe Training

In [ ]:
import glob
import cv2
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from skimage.morphology import skeletonize
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split

DEEPGLOBE_ROOT = '/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset'
IMG_SIZE = 512

train_transforms = A.Compose([
    A.RandomCrop(width=IMG_SIZE, height=IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15, p=0.8),
    A.ToGray(p=0.25),
    A.ToFloat(max_value=255.0),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.CenterCrop(width=IMG_SIZE, height=IMG_SIZE),
    A.ToFloat(max_value=255.0),
    ToTensorV2()
])


def compute_metrics(logits, target, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    
    
    if target.sum() == 0:
        return {'iou': 1.0 if preds.sum() == 0 else 0.0}
        
    intersection = (preds * target).sum()
    union = (preds + target).sum() - intersection
    iou = (intersection + 1e-6) / (union + 1e-6)
    return {'iou': iou.item()}


class DeepGlobeDataset(Dataset):
    def __init__(self, img_paths, transforms):
        self.img_paths = img_paths
        self.transforms = transforms
        
    def __len__(self):
        return len(self.img_paths)
        
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        mask_path = img_path.replace('_sat.jpg', '_mask.png')
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.float32)
        
        augmented = self.transforms(image=image, mask=mask)
        x_3ch = augmented['image']
        y_seg_np = augmented['mask'].numpy()
        
        
        fake_nir = x_3ch[0:1, :, :]
        x_4ch = torch.cat([x_3ch, fake_nir], dim=0)
        
        y_skel_np = skeletonize(y_seg_np > 0).astype(np.float32) if y_seg_np.sum() > 0 else np.zeros_like(y_seg_np)
        return x_4ch, torch.from_numpy(y_seg_np).unsqueeze(0), torch.from_numpy(y_skel_np).unsqueeze(0)

all_files = sorted(glob.glob(f"{DEEPGLOBE_ROOT}/train/**/*_sat.jpg", recursive=True))


train_files, val_files = train_test_split(all_files, test_size=0.1, random_state=42)

train_ds = DeepGlobeDataset(train_files, transforms=train_transforms)
val_ds = DeepGlobeDataset(val_files, transforms=val_transforms) 

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model = OccluNetLite(in_channels=4).to(DEVICE)
checkpoint = torch.load(best_ckpt_p2, map_location=DEVICE)
if 'model_state' in checkpoint:
    model.load_state_dict(checkpoint['model_state'])
else:
    model.load_state_dict(checkpoint)

enc_early, enc_late, _ = model.get_param_groups()
for param in enc_early: param.requires_grad = False
for param in enc_late: param.requires_grad = True

optimizer_dg = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion_dg = FullLoss(phase=2).to(DEVICE)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

best_val_iou_dg = 0.0
best_ckpt_dg = f'{OUTPUT_DIR}/best_deepglobe_transfer.pth'


print(f"\nStarting DeepGlobe Training ({len(train_files)} train, {len(val_files)} val)...")

for epoch in range(1, 16):
    model.train()
    running_loss = 0.0
    
    tbar = tqdm(train_loader, desc=f'Train Ep{epoch:02d}', leave=False)
    for x, y_seg, y_skel in tbar:
        x, y_seg, y_skel = x.to(DEVICE), y_seg.to(DEVICE), y_skel.to(DEVICE)
        optimizer_dg.zero_grad()
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                seg_logits, skel_logits = model(x)
                loss, _ = criterion_dg(seg_logits, skel_logits, y_seg, y_skel)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer_dg)
            scaler.update()
        else:
            seg_logits, skel_logits = model(x)
            loss, _ = criterion_dg(seg_logits, skel_logits, y_seg, y_skel)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_dg.step()
            
        running_loss += loss.item()
        tbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    val_iou_sum = 0
    with torch.no_grad():
        for x, y_seg, y_skel in tqdm(val_loader, desc=f'Val Ep{epoch:02d}', leave=False):
            x, y_seg, y_skel = x.to(DEVICE), y_seg.to(DEVICE), y_skel.to(DEVICE)
            seg_logits, skel_logits = model(x)
            loss, _ = criterion_dg(seg_logits, skel_logits, y_seg, y_skel)
            val_loss += loss.item()
            
            m = compute_metrics(seg_logits, y_seg, threshold=0.5)
            val_iou_sum += m['iou']
            
    avg_val_loss = val_loss / len(val_loader)
    avg_val_iou = val_iou_sum / len(val_loader)
    
    status = "(New Best)" if avg_val_iou > best_val_iou_dg else ""
    if avg_val_iou > best_val_iou_dg:
        best_val_iou_dg = avg_val_iou
        torch.save({'model_state': model.state_dict(), 'val_iou': avg_val_iou}, best_ckpt_dg)
        
    print(f'Ep {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val IoU: {avg_val_iou:.4f} {status}')

## Phase 3: Sentinel-2 Domain Adaptation (optional, 20 epochs)

In [ ]:
import os
import glob
import cv2
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
from skimage.morphology import skeletonize
from tqdm import tqdm

def compute_metrics_robust(logits, target, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    
    if target.sum() == 0:
        return {'iou': 1.0 if preds.sum() == 0 else 0.0}
        
    intersection = (preds * target).sum()
    union = (preds + target).sum() - intersection
    iou = (intersection + 1e-6) / (union + 1e-6)
    return {'iou': iou.item()}

class Sentinel2Dataset(Dataset):
    """Loads RGB+NIR images and pre-rendered masks directly from Kaggle folders."""
    def __init__(self, root, img_size=512, use_tif=False):
        self.img_size = img_size
        
        if use_tif:
            img_dir = os.path.join(root, 'images_tif')
            mask_dir = os.path.join(root, 'masks_tif', 'masks_tif') 
            ext = '.tif'
        else:
            img_dir = os.path.join(root, 'images_png')
            mask_dir = os.path.join(root, 'masks_png', 'masks_png') 
            ext = '.png'
            
        all_imgs = sorted(glob.glob(os.path.join(img_dir, f'*{ext}')))
        self.pairs = []
        
        for img_path in all_imgs:
            filename = os.path.basename(img_path)
            mask_path = os.path.join(mask_dir, filename)
            
            if os.path.exists(mask_path):
                self.pairs.append((img_path, mask_path))
                
        print(f'Sentinel-2 dataset loaded: {len(self.pairs)} exact image-mask pairs found.')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        
        x_3ch = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        
        fake_nir = x_3ch[0:1, :, :]
        x_4ch = torch.cat([x_3ch, fake_nir], dim=0)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        y_seg_np = (mask > 127).astype(np.float32)
        y_seg = torch.from_numpy(y_seg_np).unsqueeze(0)
        
        y_skel_np = skeletonize(y_seg_np > 0).astype(np.float32) if y_seg_np.sum() > 0 else np.zeros_like(y_seg_np)
        y_skel = torch.from_numpy(y_skel_np).unsqueeze(0)
        
        return x_4ch, y_seg, y_skel

if os.path.exists(SENTINEL_ROOT):
    try:
        s2_ds = Sentinel2Dataset(SENTINEL_ROOT, img_size=IMG_SIZE, use_tif=False)
        
        if len(s2_ds) > 0:
            print(f'\nStarting Phase 3: Domain Adaptation & Fine-Tuning...')
            
            print(f"Loading sequential weights from: {best_ckpt_dg}")
            ckpt = torch.load(best_ckpt_dg, map_location=DEVICE)
            model.load_state_dict(ckpt['model_state'] if 'model_state' in ckpt else ckpt)
            
            
            n_s2_use = max(1, len(s2_ds) // 5)
            s2_sub, _ = random_split(s2_ds, [n_s2_use, len(s2_ds) - n_s2_use])
            
            mixed_ds = ConcatDataset([train_ds, s2_sub])
            mixed_loader = DataLoader(mixed_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
            
            
            optimizer_p3 = optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
            criterion_p3 = FullLoss(phase=2).to(DEVICE) 
            best_ckpt_p3 = f'{OUTPUT_DIR}/best_phase3.pth'
            best_val_iou_p3 = 0.0
            
            for epoch in range(1, PHASE3_EPOCHS + 1):
                model.train()
                running_loss = 0.0
                
                tbar = tqdm(mixed_loader, desc=f'P3 Ep{epoch:02d} Train', leave=False)
                for x, y_seg, y_skel in tbar:
                    x, y_seg, y_skel = x.to(DEVICE), y_seg.to(DEVICE), y_skel.to(DEVICE)
                    optimizer_p3.zero_grad()
                    
                    seg_logits, skel_logits = model(x)
                    loss, _ = criterion_p3(seg_logits, skel_logits, y_seg, y_skel)
                    loss.backward()
                    optimizer_p3.step()
                    
                    running_loss += loss.item()
                    tbar.set_postfix(loss=f"{loss.item():.4f}")
                
                train_loss = running_loss / len(mixed_loader)
                
                
                model.eval()
                val_ious = []
                with torch.no_grad():
                    for x, y_seg, _ in tqdm(DataLoader(s2_ds, batch_size=2), desc='S2 Val', leave=False):
                        x, y_seg = x.to(DEVICE), y_seg.to(DEVICE)
                        seg_logits, _ = model(x)
                        
                        m = compute_metrics_robust(seg_logits, y_seg, threshold=0.5)
                        val_ious.append(m['iou'])
                
                avg_iou = np.mean([i for i in val_ious if i >= 0.0]) if val_ious else 0.0
                
                status = "(New Best)" if avg_iou > best_val_iou_p3 else ""
                if avg_iou > best_val_iou_p3:
                    best_val_iou_p3 = avg_iou
                    torch.save({'model_state': model.state_dict(), 'iou': avg_iou}, best_ckpt_p3)
                    
                print(f'P3 Ep{epoch:02d} | Train Loss: {train_loss:.4f} | S2 Val IoU: {avg_iou:.4f} {status}')

            print(f'\nPhase 3 Complete. Best Checkpoint saved to: {best_ckpt_p3}')
        else:
            print('Sentinel-2 pairs not found. Check the SENTINEL_ROOT path.')
            
    except Exception as e:
        print(f'Phase 3 Error: {e}')
else:
    print('SENTINEL_ROOT path does not exist. Skipping Phase 3.')

## Bengaluru Validation (Sentinel-2 Inference)

In [ ]:
import glob as glob_mod

ckpt_files = sorted(glob_mod.glob(f'{OUTPUT_DIR}/best_phase*.pth'), reverse=True)
if ckpt_files:
    best_final_ckpt = ckpt_files[0]
    ckpt = torch.load(best_final_ckpt, map_location=DEVICE)
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
    else:
        model.load_state_dict(ckpt)
    print(f'Loaded: {best_final_ckpt}')
else:
    print('No checkpoint found — using current model weights')

model.eval()

if os.path.exists(SENTINEL_ROOT):
    s2_val_ds = Sentinel2Dataset(SENTINEL_ROOT, img_size=IMG_SIZE)
    s2_val_loader = DataLoader(s2_val_ds, batch_size=2, shuffle=False,
                               num_workers=NUM_WORKERS)
    
    all_ious = []
    sample_preds = []
    
    with torch.no_grad():
        for batch_idx, (x, y_seg, tif_paths) in enumerate(
                tqdm(s2_val_loader, desc='Bengaluru validation')):
            x = x.to(DEVICE)
            y_seg = y_seg.to(DEVICE)
            seg_logits, skel_logits = model(x)
            
            if y_seg.sum() > 0:  
                m = compute_metrics(seg_logits, y_seg)
                all_ious.append(m['iou'])
            
            
            if batch_idx < 3:
                sample_preds.append({
                    'rgb':  x[0, :3].cpu().permute(1,2,0).numpy().clip(0,1),
                    'seg':  torch.sigmoid(seg_logits[0, 0]).cpu().numpy(),
                    'skel': torch.sigmoid(skel_logits[0, 0]).cpu().numpy(),
                })
    
    if all_ious:
        print(f'\nBengaluru val IoU: {np.mean(all_ious):.4f} ± {np.std(all_ious):.4f}')
        print(f'(Range: {min(all_ious):.4f} – {max(all_ious):.4f})')
    
    
    fig, axes = plt.subplots(len(sample_preds), 3, figsize=(15, 5*len(sample_preds)))
    if len(sample_preds) == 1:
        axes = [axes]
    
    for i, pred in enumerate(sample_preds):
        axes[i][0].imshow(pred['rgb'])
        axes[i][0].set_title('Bengaluru RGB')
        axes[i][0].axis('off')
        axes[i][1].imshow(pred['seg'], cmap='Reds', vmin=0, vmax=1)
        axes[i][1].set_title('Seg Prediction')
        axes[i][1].axis('off')
        axes[i][2].imshow(pred['skel'], cmap='Blues', vmin=0, vmax=1)
        axes[i][2].set_title('Skeleton Prediction')
        axes[i][2].axis('off')
    
    plt.suptitle('Bengaluru (Sentinel-2) Predictions', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/bengaluru_predictions.png', dpi=100)
    plt.show()
else:
    print('SENTINEL_ROOT not found — Bengaluru validation skipped')

## Graph Pipeline (Skeletonize → sknw → MST → Centrality → Resilience)

In [ ]:
import networkx as nx
import sknw
from skimage.morphology import skeletonize
from scipy.spatial.distance import cdist
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm


class RoadGraphExtractor:
    """
    Full graph pipeline:
    1. Skeletonize seg mask + merge with skeleton head output
    2. sknw graph construction (nodes at junctions/endpoints)
    3. MST gap healing (Disjoint Set Union-Find)
    4. Betweenness centrality (gatekeeper nodes)
    5. Node ablation + Resilience Index
    
    Directly implements all PS4 graph requirements.
    """
    
    def __init__(self, connect_radius=50, angle_threshold_deg=45):
        self.connect_radius = connect_radius
        self.angle_threshold = np.radians(angle_threshold_deg)
    
    def extract_skeleton(self, seg_mask, skel_mask=None, seg_threshold=0.5,
                         skel_threshold=0.7): # Bumped threshold to 0.7 for safety
        """Step 1: Get 1-pixel skeleton from seg mask + optional skeleton head."""
        if isinstance(seg_mask, torch.Tensor):
            seg_mask = seg_mask.cpu().numpy()
        
        
        seg_binary = (seg_mask > seg_threshold).astype(np.uint8)
        
    
        if seg_binary.sum() == 0:
            return np.zeros_like(seg_binary)
            
        skeleton = skeletonize(seg_binary)

        if skel_mask is not None:
            if isinstance(skel_mask, torch.Tensor):
                skel_mask = skel_mask.cpu().numpy()
            skel_binary = (skel_mask > skel_threshold).astype(np.uint8)
            
            
            skel_binary = skel_binary * seg_binary 
            
            skel_head_thin = skeletonize(skel_binary)
            skeleton = np.logical_or(skeleton, skel_head_thin).astype(np.uint8)
        
        return skeleton
    
    def build_graph(self, skeleton):
        """Step 2: sknw graph construction."""
        if skeleton.sum() == 0:
            return nx.Graph()
        
        
        skel_uint8 = (skeleton > 0).astype(np.uint8)
        G = sknw.build_sknw(skel_uint8)
        
        
        for u, v, data in G.edges(data=True):
            pts = data.get('pts', np.array([[0,0]]))
            if len(pts) > 1:
                diffs = np.diff(pts, axis=0)
                length = np.sum(np.sqrt((diffs**2).sum(axis=1)))
            else:
                length = 1.0
            data['weight'] = max(length, 1.0)
        
        return G
    
    def mst_gap_healing(self, G, skeleton_shape):
        """
        Step 3: MST gap healing with Disjoint Set Union-Find.
        Bridges disconnected components via minimum-cost edges
        with angular constraint.
        """
        if G.number_of_nodes() == 0:
            return G, {'before': 0, 'after': 0, 'ratio': 0.0}
        
        n_components_before = nx.number_connected_components(G)
        
        
        endpoints = [n for n in G.nodes() if G.degree(n) == 1]
        
        if len(endpoints) < 2:
            return G, {'before': n_components_before,
                       'after': n_components_before, 'ratio': 0.0}
        
        
        pos = np.array([G.nodes[n]['o'] for n in endpoints])  # sknw node positions
        
        dists = cdist(pos, pos)
        np.fill_diagonal(dists, np.inf)

        candidate_edges = []
        for i in range(len(endpoints)):
            for j in range(i+1, len(endpoints)):
                if dists[i, j] <= self.connect_radius:
                    ni, nj = endpoints[i], endpoints[j]
                    if not nx.has_path(G, ni, nj):
                        candidate_edges.append((dists[i, j], ni, nj))
        
        
        candidate_edges.sort()
        G_healed = G.copy()
        
        for dist, ni, nj in candidate_edges:
            if not nx.has_path(G_healed, ni, nj):
                G_healed.add_edge(ni, nj, weight=dist, healed=True)
        
        n_components_after = nx.number_connected_components(G_healed)
        improvement = (n_components_before - n_components_after) / \
                       max(n_components_before, 1) * 100
        
        stats = {
            'before': n_components_before,
            'after':  n_components_after,
            'ratio':  improvement,
            'healed_edges': len(candidate_edges)
        }
        return G_healed, stats
    
    def compute_centrality(self, G, top_k=10):
        """
        Step 4: Weighted betweenness centrality.
        Identifies gatekeeper nodes (PS requirement).
        """
        if G.number_of_nodes() < 3:
            return {}, []
        
        centrality = nx.betweenness_centrality(G, weight='weight', normalized=True)
        
        gatekeepers = sorted(centrality.items(), key=lambda x: -x[1])[:top_k]
        
        return centrality, gatekeepers
    
    def global_efficiency(self, G):
        """Compute global efficiency for resilience index."""
        n = G.number_of_nodes()
        if n < 2:
            return 0.0
        
        efficiency = 0.0
        for u in G.nodes():
            lengths = nx.single_source_shortest_path_length(G, u)
            for v, d in lengths.items():
                if v != u and d > 0:
                    efficiency += 1.0 / d
        
        return efficiency / (n * (n - 1))
    
    def node_ablation(self, G, gatekeepers, max_k=10):
        """
        Step 5: Node ablation + Resilience Index.
        R = E(perturbed) / E(baseline) — PS requirement.
        """
        if G.number_of_nodes() < 3:
            return {'baseline': 0, 'resilience_curve': [], 'resilience_indices': []}
        
        E_baseline = self.global_efficiency(G)
        resilience_curve = [1.0]  
        resilience_indices = []
        
        top_nodes = [node for node, _ in gatekeepers[:max_k]]
        G_temp = G.copy()
        
        for i, node in enumerate(top_nodes):
            if node in G_temp.nodes():
                G_temp.remove_node(node)
            E_perturbed = self.global_efficiency(G_temp)
            R = E_perturbed / max(E_baseline, 1e-9)
            resilience_curve.append(R)
            resilience_indices.append({
                'k': i + 1,
                'node_removed': node,
                'R': R,
                'E_perturbed': E_perturbed
            })
        
        return {
            'baseline': E_baseline,
            'resilience_curve': resilience_curve,
            'resilience_indices': resilience_indices
        }
    
    def run_full_pipeline(self, seg_mask, skel_mask=None, top_k=10):
        """End-to-end pipeline. Returns all results."""
        skel   = self.extract_skeleton(seg_mask, skel_mask)
        G_raw  = self.build_graph(skel)
        G, mst_stats = self.mst_gap_healing(G_raw, skel.shape)
        centrality, gatekeepers = self.compute_centrality(G, top_k)
        ablation = self.node_ablation(G, gatekeepers, max_k=top_k)
        
        return {
            'skeleton': skel,
            'graph': G,
            'mst_stats': mst_stats,
            'centrality': centrality,
            'gatekeepers': gatekeepers,
            'ablation': ablation
        }


def visualize_graph_results(rgb, seg_prob, results, output_dir):
    """5-panel visualization: Input, Seg, Skeleton, Centrality, Resilience."""
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    axes[0].imshow(rgb.clip(0, 1))
    axes[0].set_title('Input Chip')
    axes[0].axis('off')
    
    axes[1].imshow(seg_prob, cmap='Reds', vmin=0, vmax=1)
    axes[1].set_title('Road Segmentation')
    axes[1].axis('off')
    
    skel = results['skeleton']
    G = results['graph']
    axes[2].imshow(skel, cmap='gray')
    if G.number_of_edges() > 0:
        for u, v, data in G.edges(data=True):
            if 'pts' in data:
                pts = data['pts']
                color = 'lime' if data.get('healed', False) else 'cyan'
                axes[2].plot(pts[:, 1], pts[:, 0], color=color, lw=0.5, alpha=0.7)
    axes[2].set_title('Skeleton + Graph\n(cyan=orig, lime=healed)')
    axes[2].axis('off')
    
    centrality = results['centrality']
    heatmap = np.zeros_like(skel, dtype=float)
    for node, score in centrality.items():
        if node in G.nodes():
            pos = G.nodes[node].get('o', None)
            if pos is not None:
                r, c = int(pos[0]), int(pos[1])
                if 0 <= r < heatmap.shape[0] and 0 <= c < heatmap.shape[1]:
                    heatmap[r, c] = score
    axes[3].imshow(rgb.clip(0,1))
    if heatmap.max() > 0:
        axes[3].imshow(heatmap, cmap='hot', alpha=0.6, vmin=0, vmax=heatmap.max())
    axes[3].set_title('Betweenness Centrality\n(Gatekeeper Nodes)')
    axes[3].axis('off')
    
    
    ablation = results['ablation']
    curve = ablation.get('resilience_curve', [1.0])
    k_vals = list(range(len(curve)))
    axes[4].plot(k_vals, curve, 'b-o', markersize=5)
    axes[4].axhline(0.8, color='orange', linestyle='--', label='R=0.8 threshold')
    axes[4].set_xlabel('Nodes Ablated (k)')
    axes[4].set_ylabel('Resilience Index R')
    axes[4].set_title('Network Resilience\nR = E(perturbed)/E(baseline)')
    axes[4].set_ylim(0, 1.05)
    axes[4].legend()
    axes[4].grid(True, alpha=0.3)
    
    plt.suptitle(f'GT-OccluNet: Full PS4 Pipeline\n'
                 f'Components: {results["mst_stats"]["before"]}→{results["mst_stats"]["after"]} | '
                 f'Connectivity: +{results["mst_stats"]["ratio"]:.1f}%',
                 fontsize=12)
    plt.tight_layout()
    save_path = f'{output_dir}/full_pipeline_result.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

extractor = RoadGraphExtractor(connect_radius=50)

model.eval()
with torch.no_grad():
    x_test, y_seg_test, y_skel_test = next(iter(val_loader))
    x_test = x_test.to(DEVICE)
    seg_logits, skel_logits = model(x_test)
    
    seg_prob_np  = torch.sigmoid(seg_logits[0, 0]).cpu().numpy()
    skel_prob_np = torch.sigmoid(skel_logits[0, 0]).cpu().numpy()
    rgb_np = x_test[0, :3].cpu().permute(1, 2, 0).numpy()

results = extractor.run_full_pipeline(seg_prob_np, skel_prob_np, top_k=10)

print('\n=== Graph Pipeline Results ===')
print('\n=== Graph Pipeline Results ===')
print(f'Skeleton pixels: {results["skeleton"].sum()}')
print(f'Graph nodes: {results["graph"].number_of_nodes()}')
print(f'Graph edges: {results["graph"].number_of_edges()}')
print(f'Connected components: {results["mst_stats"]["before"]} → {results["mst_stats"]["after"]}')
print(f'Connectivity improvement: +{results["mst_stats"]["ratio"]:.1f}%')

print(f'\nTop 5 Gatekeeper Nodes (betweenness centrality):')
if results['gatekeepers']:
    for node, score in results['gatekeepers'][:5]:
        print(f'  Node {node}: centrality={score:.4f}')
else:
    print('  None (Graph empty or too small)')

curve = results["ablation"]["resilience_curve"]
if len(curve) > 0:
    idx = min(5, len(curve) - 1)
    print(f'\nResilience Index after k={idx} ablations: {curve[idx]:.4f}')
else:
    print('\nResilience Index: N/A (Graph empty)')

visualize_graph_results(rgb_np, seg_prob_np, results, OUTPUT_DIR)

## Gradio Dashboard (5 panels)

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import torch
import cv2
import io
from PIL import Image

model.eval()
extractor_app = RoadGraphExtractor(connect_radius=50)


def run_pipeline(image, seg_threshold=0.5, n_ablate=5):
    """
    Full inference pipeline for Gradio.
    Returns 5 panel images.
    """
    if image is None:
        return None, None, None, None, None
    

    img_arr = np.array(image)
    if img_arr.shape[-1] == 4:  
        img_arr = img_arr[..., :3]
    
    img_resized = cv2.resize(img_arr, (512, 512))
    
    
    chip_4ch = np.stack([
        img_resized[..., 0].astype(np.float32),  
        img_resized[..., 1].astype(np.float32),  
        img_resized[..., 2].astype(np.float32),  
        img_resized[..., 0].astype(np.float32),  
    ], axis=-1)
    
    chip_norm = percentile_stretch(chip_4ch)
    x_tensor = torch.from_numpy(chip_norm.transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        seg_logits, skel_logits = model(x_tensor)
    
    seg_prob  = torch.sigmoid(seg_logits[0, 0]).cpu().numpy()
    skel_prob = torch.sigmoid(skel_logits[0, 0]).cpu().numpy()

    results = extractor_app.run_full_pipeline(
        seg_prob, skel_prob, top_k=max(n_ablate, 5)
    )
    
    rgb_display = chip_norm[..., :3]
    
    def to_pil(arr, cmap='viridis', alpha_base=None):
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(arr, cmap=cmap, vmin=0, vmax=arr.max() if arr.max() > 0 else 1)
        ax.axis('off')
        plt.tight_layout(pad=0)
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close()
        buf.seek(0)
        return Image.open(buf).copy()
    

    p1 = Image.fromarray((rgb_display * 255).clip(0, 255).astype(np.uint8))
    

    p2 = to_pil(seg_prob > seg_threshold, cmap='Reds')
    

    fig3, ax3 = plt.subplots(figsize=(5, 5))
    ax3.imshow(rgb_display.clip(0, 1))
    skel = results['skeleton']
    G = results['graph']

    for u, v, data in G.edges(data=True):
        if 'pts' in data:
            pts = data['pts']
            color = 'lime' if data.get('healed', False) else 'cyan'
            ax3.plot(pts[:, 1], pts[:, 0], color=color, lw=1.0, alpha=0.8)
    ax3.set_title(f'Components: {results["mst_stats"]["before"]}→{results["mst_stats"]["after"]}')
    ax3.axis('off')
    plt.tight_layout(pad=0)
    buf3 = io.BytesIO()
    plt.savefig(buf3, format='png', dpi=100, bbox_inches='tight')
    plt.close()
    buf3.seek(0)
    p3 = Image.open(buf3).copy()
    
    centrality = results['centrality']
    heatmap = np.zeros_like(skel, dtype=float)
    for node, score in centrality.items():
        if node in G.nodes():
            pos = G.nodes[node].get('o', None)
            if pos is not None:
                r, c = int(pos[0]), int(pos[1])
                if 0 <= r < heatmap.shape[0] and 0 <= c < heatmap.shape[1]:
                
                    rr = slice(max(0,r-3), min(heatmap.shape[0],r+4))
                    cc = slice(max(0,c-3), min(heatmap.shape[1],c+4))
                    heatmap[rr, cc] = max(heatmap[rr, cc].max(), score)
    
    fig4, ax4 = plt.subplots(figsize=(5, 5))
    ax4.imshow(rgb_display.clip(0, 1))
    if heatmap.max() > 0:
        im = ax4.imshow(heatmap, cmap='hot', alpha=0.7,
                         vmin=0, vmax=heatmap.max())
        plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)
    top5_nodes = results['gatekeepers'][:5]
    for node, score in top5_nodes:
        if node in G.nodes():
            pos = G.nodes[node].get('o', None)
            if pos is not None:
                ax4.plot(pos[1], pos[0], 'y*', markersize=12)
    ax4.set_title('Betweenness Centrality Heatmap')
    ax4.axis('off')
    plt.tight_layout(pad=0)
    buf4 = io.BytesIO()
    plt.savefig(buf4, format='png', dpi=100, bbox_inches='tight')
    plt.close()
    buf4.seek(0)
    p4 = Image.open(buf4).copy()
    
    ablation = results['ablation']
    curve = ablation.get('resilience_curve', [1.0])
    k_vals = list(range(len(curve)))
    
    fig5, ax5 = plt.subplots(figsize=(5, 5))
    ax5.plot(k_vals, curve, 'b-o', markersize=6, linewidth=2)
    
    ablate_idx = min(n_ablate, len(curve)-1)
    ax5.axvline(x=ablate_idx, color='red', linestyle='--', alpha=0.7,
                 label=f'k={n_ablate}')
    if ablate_idx < len(curve):
        ax5.plot(ablate_idx, curve[ablate_idx], 'r*', markersize=15)
    ax5.axhline(0.8, color='orange', linestyle=':', label='R=0.8')
    ax5.set_xlabel('Nodes Ablated (k)', fontsize=12)
    ax5.set_ylabel('Resilience Index R', fontsize=12)
    ax5.set_title(f'Network Resilience Curve\nR(k={n_ablate}) = {curve[ablate_idx]:.3f}',
                   fontsize=12)
    ax5.set_ylim(0, 1.05)
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    plt.tight_layout()
    buf5 = io.BytesIO()
    plt.savefig(buf5, format='png', dpi=100, bbox_inches='tight')
    plt.close()
    buf5.seek(0)
    p5 = Image.open(buf5).copy()
    
    return p1, p2, p3, p4, p5


with gr.Blocks(title='GT-OccluNet: Route Resilience') as demo:
    gr.Markdown("""
    # GT-OccluNet: Occlusion-Robust Road Extraction & Network Resilience
    ** ResNet34 + CBAM + Dilated Bottleneck + RCPM + Dual Heads**
    
    Upload a satellite image chip (512×512 preferred). The system will:
    1. Extract road segmentation mask + skeleton centerline
    2. Build topological graph via sknw + MST gap healing  
    3. Compute betweenness centrality (gatekeeper nodes)
    4. Run node ablation → Resilience Index curve
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label='Upload Satellite Chip', type='pil')
            seg_threshold = gr.Slider(
                minimum=0.3, maximum=0.7, value=0.5, step=0.05,
                label='Detection Threshold'
            )
            n_ablate = gr.Slider(
                minimum=1, maximum=10, value=5, step=1,
                label='Nodes to Ablate (k)'
            )
            run_btn = gr.Button('Run Pipeline', variant='primary')
        
        with gr.Column(scale=4):
            gr.Markdown('### Results')
            with gr.Row():
                out1 = gr.Image(label='Panel 1: Input Chip')
                out2 = gr.Image(label='Panel 2: Road Segmentation')
                out3 = gr.Image(label='Panel 3: Skeleton + Healed Graph')
            with gr.Row():
                out4 = gr.Image(label='Panel 4: Criticality Heatmap (Betweenness Centrality)')
                out5 = gr.Image(label='Panel 5: Resilience Curve (Node Ablation)')
    
    run_btn.click(
        fn=run_pipeline,
        inputs=[input_image, seg_threshold, n_ablate],
        outputs=[out1, out2, out3, out4, out5]
    )
    
    gr.Markdown("""
    ---
    **Panel Guide:**
    - Panel 2: Thresholded road segmentation mask  
    - Panel 3: Skeleton + graph edges (cyan=original, lime=MST-healed gaps)  
    - Panel 4: Betweenness centrality heatmap — stars = top-5 gatekeeper nodes  
    - Panel 5: R = E(perturbed)/E(baseline) — network degradation curve  
    """)

demo.launch(share=True, debug=False)

## Training Curves & Final Report

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if history['train_loss']:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    epochs_p1 = list(range(1, len(history['train_loss']) + 1))
    
    axes[0].plot(epochs_p1, history['train_loss'], 'b-', label='Train Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Phase 1: Training Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs_p1, history['val_iou'],   'g-',  label='Val IoU')
    axes[1].plot(epochs_p1, history['val_dice'],  'b--', label='Val Dice')
    axes[1].plot(epochs_p1, history['val_recall'],'r:',  label='Val Recall')
    axes[1].axhline(0.40, color='orange', linestyle='--', alpha=0.7, label='IoU=0.40 target')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
    axes[1].set_title('Phase 1: Validation Metrics')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    if history_p2['train_loss']:
        all_iou = history['val_iou'] + history_p2['val_iou']
        epochs_all = list(range(1, len(all_iou) + 1))
        axes[2].plot(epochs_all, all_iou, 'g-', label='Val IoU (all phases)')
        axes[2].axvline(x=PHASE1_EPOCHS, color='red', linestyle='--', 
                         alpha=0.5, label='Phase 2 start')
        axes[2].axhline(0.48, color='orange', linestyle='--', 
                         alpha=0.7, label='IoU=0.48 target')
        axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val IoU')
        axes[2].set_title('Combined Training Curve')
        axes[2].legend(); axes[2].grid(True, alpha=0.3)
    
    plt.suptitle('GT-OccluNet-Lite Training History', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120)
    plt.show()

print('=' * 60)
print('GT-OccluNet: Final Results Summary')
print('=' * 60)
print(f'Architecture: ResNet34 + CBAM + Dilated Bottleneck (r=1,2,4,8)')
print(f'             + RCPM (BiGRU path continuity at 64x64)')
print(f'             + Dual Heads (Seg + Skeleton)')
print(f'Input: 4-channel (RGB + NIR)')
print(f'Loss: Dice + WBCE(pw=15) + CLDice + Conformity')
print()
if history['val_iou']:
    print(f'Phase 1 best val IoU:  {best_val_iou:.4f}')
if hasattr(best_val_iou_p2, 'item') or isinstance(best_val_iou_p2, float):
    if best_val_iou_p2 > 0:
        print(f'Phase 2 best val IoU:  {best_val_iou_p2:.4f}')
print()
print('Graph Pipeline:')
if 'mst_stats' in results:
    print(f'  Components: {results["mst_stats"]["before"]} → {results["mst_stats"]["after"]}')
    print(f'  Connectivity improvement: +{results["mst_stats"]["ratio"]:.1f}%')
    print(f'  Healed edges: {results["mst_stats"].get("healed_edges", 0)}')
if 'gatekeepers' in results and results['gatekeepers']:
    print(f'  Top gatekeeper centrality: {results["gatekeepers"][0][1]:.4f}')
if 'ablation' in results:
    curve = results['ablation'].get('resilience_curve', [1.0])
    if len(curve) > 5:
        print(f'  Resilience at k=5: {curve[5]:.4f}')
print()
print('Compliance:')
ps4_items = [
    ('Attention mechanism (spatial + channel)', 'CBAM at each encoder scale'),
    ('Occlusion handling', 'Dilated bottleneck (r=8) + RCPM + occlusion aug'),
    ('Long-range context', 'Dilated bottleneck + BiGRU path continuity'),
    ('Multi-scale features', 'UNet skip connections (4 scales)'),
    ('Topological reconstruction', 'sknw + MST gap healing'),
    ('Betweenness centrality', 'NetworkX weighted betweenness'),
    ('Gatekeeper nodes', 'Top-k centrality nodes'),
    ('Node ablation', 'Sequential removal with efficiency recompute'),
    ('Resilience Index', 'R = E(perturbed)/E(baseline)'),
    ('Visualization dashboard', 'Gradio 5-panel with 2 sliders'),
    ('SpaceNet training data', 'Mumbai PS-MS chips (RGB + NIR)'),
    ('Indian city validation', 'Bengaluru Sentinel-2'),
]
for req, impl in ps4_items:
    print(f'   {req}')
    print(f'     → {impl}')

print()
print(f'Checkpoints saved to: {OUTPUT_DIR}/')
print(f'Best model: best_phase2.pth')
print('=' * 60)